<a href="https://colab.research.google.com/github/the-LALEL/danny-veo-test/blob/main/quickstarts/Get_started_Deep_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Deep Research Agent
<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_Deep_Research.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Deep Research Agent autonomously plans, executes, and synthesizes multi-step research tasks into detailed, cited reports. Powered by Gemini, it navigates complex information landscapes — searching the web, reading pages, executing code, and analyzing documents — to produce comprehensive reports in minutes.

Deep Research is ideal for market analysis, competitive intelligence, literature reviews, and technical deep dives where you need more than a simple answer. It can consult over 100 sources in a single task.

Deep Research is exclusively available through the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) and cannot be accessed via `generate_content`. Research tasks run asynchronously in the background because they can take several minutes to complete.

The [documentation](https://ai.google.dev/gemini-api/docs/deep-research) is a good place to start learning more about the Deep Research agent.

## Setup

### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [2]:
from google.colab import userdata

GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')

SecretNotFoundError: Secret GEMINI_API_KEY does not exist.

### Install and initialize the SDK

In [ ]:
%pip install -U -q "google-genai>=1.73.0"

In [ ]:
import time
import base64
from google import genai
from IPython.display import display, Image, Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

### Select a Deep Research agent

Two versions of the Deep Research agent are available:

- **Deep Research** (`deep-research-preview-04-2026`): Optimized for speed and efficiency with reduced latency. Ideal for interactive use cases.
- **Deep Research Max** (`deep-research-max-preview-04-2026`): Designed for maximum search exhaustiveness and report comprehensiveness. Best for automated reporting.

In [ ]:
AGENT = "deep-research-preview-04-2026" # @param ["deep-research-preview-04-2026","deep-research-max-preview-04-2026"] {"allow-input":true, isTemplate: true}

Next, create helper functions to poll for results and display outputs:

In [ ]:
# @title Helper functions (just run this cell)

def wait_for_result(interaction, poll_interval=10):
    """Poll until a background interaction completes or fails."""
    print(f"Research started: {interaction.id}")
    while True:
        result = client.interactions.get(interaction.id)
        if result.status == "completed":
            return result
        elif result.status == "failed":
            print(f"Research failed: {result.error}")
            return result
        print(f"  Status: {result.status}", flush=True)
        time.sleep(poll_interval)


def display_outputs(result):
    """Display text and image outputs from a completed interaction."""
    for output in result.outputs:
        if output.type == "text":
            display(Markdown(output.text))
        elif output.type == "image" and output.data:
            image_bytes = base64.b64decode(output.data)
            display(Image(data=image_bytes))

## Run your first Deep Research task

Start a research task with `background=True` and poll for the result. Deep Research is asynchronous — tasks can take several minutes as the agent plans, searches, reads, and synthesizes information.

In [ ]:
interaction = client.interactions.create(
    input="Research the history of Google TPUs and their impact on AI development.",
    agent=AGENT,
    background=True,
)

result = wait_for_result(interaction)
display_outputs(result)

## Collaborative planning

Collaborative planning lets you review and refine the research plan before the agent starts its work. When enabled, the agent returns a proposed plan instead of executing immediately. You can iterate on the plan through multi-turn interactions.

### Step 1: Request a plan

Set `collaborative_planning=True` in the `agent_config`. The agent returns a research plan instead of a full report.

In [ ]:
plan_interaction = client.interactions.create(
    agent=AGENT,
    input="Research Google TPUs vs competitor AI accelerator hardware.",
    agent_config={
        "type": "deep-research",
        "collaborative_planning": True,
    },
    background=True,
)

result = wait_for_result(plan_interaction)
display_outputs(result)

### Step 2: Refine the plan (optional)

Use `previous_interaction_id` to continue the conversation and iterate on the plan. Keep `collaborative_planning=True` to stay in planning mode. You can repeat this step as many times as needed.

In [ ]:
refined_plan = client.interactions.create(
    agent=AGENT,
    input="Add a section comparing power efficiency and total cost of ownership.",
    agent_config={
        "type": "deep-research",
        "collaborative_planning": True,
    },
    previous_interaction_id=plan_interaction.id,
    background=True,
)

result = wait_for_result(refined_plan)
display_outputs(result)

### Step 3: Approve and execute

Set `collaborative_planning=False` to approve the plan and start the full research. The agent will execute the plan and return the final report.

> **Important:** You must explicitly set `collaborative_planning=False` on the final turn. Simply sending "go ahead" without flipping the flag will not trigger report generation.

In [ ]:
final_report = client.interactions.create(
    agent=AGENT,
    input="Plan looks good!",
    agent_config={
        "type": "deep-research",
        "collaborative_planning": False,
    },
    previous_interaction_id=refined_plan.id,
    background=True,
)

result = wait_for_result(final_report)
display_outputs(result)

## Built-in charts and infographics

Set `visualization="auto"` in the `agent_config` to enable agent-generated charts, graphs, and infographics. For best results, explicitly ask for visuals in your query — for example, "Include charts showing trends" or "Generate graphics comparing market share."

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="Analyze global semiconductor market trends from 2020 to 2025. Include charts showing market share changes between the top vendors.",
    agent_config={
        "type": "deep-research",
        "visualization": "auto",
    },
    background=True,
)

result = wait_for_result(interaction)
display_outputs(result)

> **Tip:** Setting `visualization="auto"` enables the capability, but the agent generates visuals only when the prompt requests them. Be explicit about what charts or graphics you want.

## Real-time streaming

Stream research progress in real time instead of waiting for the final result. Set `stream=True` alongside `background=True`. Enable `thinking_summaries="auto"` to see the agent's intermediate reasoning steps as it works.

### Stream event types

| Event type | Delta type | Description |
| :--- | :--- | :--- |
| `interaction.start` | — | Provides the interaction ID |
| `content.delta` | `thought_summary` | Intermediate reasoning step |
| `content.delta` | `text` | Part of the final text output |
| `content.delta` | `image` | A generated image (base64-encoded) |
| `interaction.complete` | — | Research is finished |

In [ ]:
stream = client.interactions.create(
    input="Research AI chip market trends. Include charts comparing major vendors.",
    agent=AGENT,
    background=True,
    stream=True,
    agent_config={
        "type": "deep-research",
        "thinking_summaries": "auto",
        "visualization": "auto",
    },
)

interaction_id = None
last_event_id = None
is_complete = False

def process_stream(stream):
    global interaction_id, last_event_id, is_complete
    for chunk in stream:
        if chunk.event_type == "interaction.start":
            interaction_id = chunk.interaction.id
            print(f"Interaction started: {interaction_id}")
        if chunk.event_id:
            last_event_id = chunk.event_id
        if chunk.event_type == "content.delta":
            if chunk.delta.type == "text":
                print(chunk.delta.text, end="", flush=True)
            elif chunk.delta.type == "thought_summary":
                print(f"\n💭 {chunk.delta.content.text}", flush=True)
            elif chunk.delta.type == "image" and chunk.delta.data:
                image_bytes = base64.b64decode(chunk.delta.data)
                display(Image(data=image_bytes))
        elif chunk.event_type in ("interaction.complete", "error"):
            is_complete = True
            if chunk.event_type == "interaction.complete":
                print("\n\n✅ Research Complete")

process_stream(stream)

# Reconnect if the connection drops
while not is_complete and interaction_id:
    status = client.interactions.get(interaction_id)
    if status.status != "in_progress":
        break
    print("\n🔄 Reconnecting...")
    stream = client.interactions.get(
        id=interaction_id, stream=True, last_event_id=last_event_id,
    )
    process_stream(stream)

## Tool configuration

By default, the agent uses Google Search, URL Context, and Code Execution. You can customize the tools by passing a `tools` list. This lets you restrict the agent to only web search, only private sources, or a mix of both.

| Tool | Type value | Default | Description |
| :--- | :--- | :--- | :--- |
| Google Search | `google_search` | ✅ | Search the public web |
| URL Context | `url_context` | ✅ | Read and summarize web pages |
| Code Execution | `code_execution` | ✅ | Run code for calculations and data analysis |
| MCP Server | `mcp_server` | — | Connect remote MCP servers |
| File Search | `file_search` | — | Search uploaded document corpora |

### Restrict to Google Search only

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What are the latest developments in quantum computing?",
    tools=[{"type": "google_search"}],
    background=True,
)

result = wait_for_result(interaction)
display_outputs(result)

### Combine multiple tools

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="Research the latest breakthroughs in fusion energy. Include data analysis of funding trends.",
    tools=[
        {"type": "google_search"},
        {"type": "url_context"},
        {"type": "code_execution"},
    ],
    background=True,
)

result = wait_for_result(interaction)
display_outputs(result)

## Remote MCP servers

Connect remote [MCP (Model Context Protocol)](https://modelcontextprotocol.io/) servers to give the agent access to external tools and services. Pass the server `name`, `url`, and optional authentication headers.

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="Check the status of my last server deployment.",
    tools=[
        {
            "type": "mcp_server",
            "name": "Deployment Tracker",
            "url": "https://mcp.example.com/mcp",
            "headers": {"Authorization": "Bearer YOUR_TOKEN"},
        }
    ],
    background=True,
)

You can use `allowed_tools` to restrict which tools the agent can call from an MCP server:

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="Get recent tickets from our issue tracker.",
    tools=[
        {
            "type": "mcp_server",
            "name": "Issue Tracker",
            "url": "https://mcp.example.com/mcp",
            "headers": {"Authorization": "Bearer YOUR_TOKEN"},
            "allowed_tools": ["list_tickets", "get_ticket"],
        }
    ],
    background=True,
)

> **Note:** MCP servers support no-auth, bearer token, and OAuth. For OAuth, fetch a token with a library like `google-auth` and pass it in `headers`.

## File Search

Give the agent access to your own data by using [File Search](https://ai.google.dev/gemini-api/docs/file-search). This lets the agent search your uploaded document corpora alongside the web.

In [ ]:
interaction = client.interactions.create(
    input="Compare our 2025 fiscal year report against current public web news.",
    agent=AGENT,
    background=True,
    tools=[
        {
            "type": "file_search",
            "file_search_store_names": ["fileSearchStores/my-store-name"],
        }
    ],
)

result = wait_for_result(interaction)
display_outputs(result)

## Multimodal inputs

Deep Research supports multimodal inputs including images and documents (PDFs). The agent analyzes the provided content and conducts web-based research contextualized by the inputs.

### Image input

Pass images alongside your text prompt to ground the agent's research in visual content.

In [ ]:
interaction = client.interactions.create(
    input=[
        {
            "type": "text",
            "text": "Identify the architectural style shown in this image. Research its historical origins, key characteristics, and notable examples worldwide.",
        },
        {
            "type": "image",
            "uri": "https://storage.googleapis.com/generativeai-downloads/images/generated_elephants_giraffes_zebras_sunset.jpg",
        },
    ],
    agent=AGENT,
    background=True,
)

result = wait_for_result(interaction)
display_outputs(result)

### Document input (PDF)

Pass documents directly as input. The agent analyzes the document and researches its content against the web.

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input=[
        {"type": "text", "text": "What has been the impact of this research paper? Who are the key authors and what have they worked on since?"},
        {
            "type": "document",
            "uri": "https://arxiv.org/pdf/1706.03762",
            "mime_type": "application/pdf",
        },
    ],
    background=True,
)

result = wait_for_result(interaction)
display_outputs(result)

## Steerability and formatting

Control the structure and tone of the output by providing explicit formatting instructions in your prompt. You can request specific sections, data tables, or adjust the tone for different audiences.

In [ ]:
prompt = """
Research the competitive landscape of EV batteries.

Format the output as a technical report with the following structure:
1. Executive Summary (max 200 words)
2. Key Players (Must include a data table comparing capacity, chemistry, and market share)
3. Technology Trends
4. Supply Chain Risks
5. Outlook for 2026-2030

Use a professional, technical tone suitable for an engineering audience.
"""

interaction = client.interactions.create(
    input=prompt,
    agent=AGENT,
    background=True,
)

result = wait_for_result(interaction)
display_outputs(result)

## Follow-up questions

After receiving a report, you can continue the conversation using `previous_interaction_id` to ask for clarification, summarization, or elaboration on specific sections without restarting the entire research task.

> **Note:** Follow-up questions use a standard Gemini model (not the Deep Research agent), so they return immediately without requiring `background=True`.

In [ ]:
# Use the interaction ID from any completed research task above
follow_up = client.interactions.create(
    input="Summarize the key findings in 3 bullet points. What was the most surprising insight?",
    model="gemini-3.1-pro-preview",
    previous_interaction_id=interaction.id,
)

display(Markdown(follow_up.outputs[-1].text))

## Agent configuration reference

Deep Research uses the `agent_config` parameter to control behavior. Here is a summary of all configurable fields:

| Field | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| `type` | `string` | Required | Must be `"deep-research"` |
| `thinking_summaries` | `string` | `"none"` | `"auto"` to receive intermediate reasoning steps during streaming |
| `visualization` | `string` | `"auto"` | `"auto"` to enable charts and images; `"off"` to disable |
| `collaborative_planning` | `boolean` | `false` | `true` to enable multi-turn plan review before research |

In [ ]:
# Example: all options enabled
interaction = client.interactions.create(
    agent=AGENT,
    input="Research the competitive landscape of cloud GPUs.",
    agent_config={
        "type": "deep-research",
        "thinking_summaries": "auto",
        "visualization": "auto",
        "collaborative_planning": False,
    },
    background=True,
    stream=True,
)

## Further reading

- [Deep Research documentation](https://ai.google.dev/gemini-api/docs/deep-research)
- [Interactions API documentation](https://ai.google.dev/gemini-api/docs/interactions)
- [Try Deep Research in Google AI Studio](https://aistudio.google.com)
- [File Search documentation](https://ai.google.dev/gemini-api/docs/file-search)
- [Gemini API pricing](https://ai.google.dev/gemini-api/docs/pricing)